# 20 api contract

Question: does one coordinate of the API answer through every layer below the schema as
`okf/api.md` sections 6 and 7 say, and does a coordinate of the thread class `any` answer on every
thread, without a JVM.

`Player.allowFlight` is the coordinate. Its call of the contract with the bridge mod stands in
`bridge/contract/players.toml`, the function of that call in `golem-game` and its meaning in
`golem-model`. No line of the bridge mod answers a call yet, so a scripted contract answers in
place of the game: `golem-devtool contract call` makes the calls against it, and `golem-devtool
api coordinate` calls the coordinate functions over it. The notebook holds the calls to the page,
then the answer, the miss and the fault of a call, the server token and its frames, without which
no call is made, and the codes that a coordinate answers in place of a value. It asks
`Server.bukkitVersion`, a coordinate of the class `any`, which needs no token and makes no call,
on every thread. What a call answers is the rule of the script, which the mod has to meet, so the
cells prove the code around the script and nothing of what Java answers.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

## Setup

The cell finds the repository, builds `golem-devtool` and prints the versions of Python, cargo
and rustc. The helpers run a subcommand and write a script into a temporary directory. The
scripted contract holds four entities: Alex, a player who may fly, Sam, a player who may not, a
zombie, whose UUID has a zero at the head of every group, and a player whose every read throws. A
second scripted contract, for the steps that open a frame inside another, holds Alex alone, with
another runtime id, on another level and with no right to fly. The cell also reads `okf/api.md`
with every run of white space joined into one blank, so that a search of the page does not hang
on where a line wraps.

In [ ]:
import json
import re
import shutil
import sys
import tempfile
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-20-"))

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"
page = " ".join((P.root / "okf" / "api.md").read_text(encoding="utf-8").split())


def devtool(*arguments, expect=0):
    result = run([DEVTOOL, *arguments], cwd=P.root, timeout=300)
    assert result.returncode == expect, "golem-devtool %s exited %d\n--- stdout\n%s--- stderr\n%s" % (
        " ".join(map(str, arguments)), result.returncode, result.stdout, result.stderr)
    return (json.loads(result.stdout) if result.stdout.strip() else None), result.stderr


def write(name, document):
    path = SCRATCH / name
    path.write_text(json.dumps(document), encoding="utf-8")
    return path


def answers(*arguments, name, steps):
    out, _ = devtool(*arguments, write(name + ".json", {"entities": ENTITIES, "inner": INNER, "steps": steps}))
    assert [step["op"] for step in out["steps"]] == [step["op"] for step in steps], out
    return [step["answer"] for step in out["steps"]], out["calls"], out["innerCalls"]


def refused(*arguments, name, document):
    _, error = devtool(*arguments, write(name + ".json", document), expect=1)
    return error.strip()


ALEX = "6f2d5b3e-1c4a-4e8b-9a7d-2b1c0e9f8a71"
SAM = "0b9a8c7d-6e5f-4a3b-8c2d-1e0f9a8b7c6d"
ZOMBIE = "03c1d2e4-05a6-04c7-08e9-00a1b2c3d4e5"
BROKEN = "0f0e0d0c-0b0a-4908-8706-050403020100"
NOBODY = "00000000-0000-4000-8000-000000000000"
FAULT = "java.lang.IllegalStateException: the test entity throws"
ENTITIES = [
    {"uuid": ALEX, "id": 7, "level": 0, "player": True, "allowFlight": True},
    {"uuid": SAM, "id": 8, "level": 1, "player": True},
    {"uuid": ZOMBIE, "id": 9, "level": 0},
    {"uuid": BROKEN, "id": 10, "level": 0, "player": True, "fault": FAULT},
]
INNER = [{"uuid": ALEX, "id": 70, "level": 2, "player": True}]
AT = {ALEX: {"id": 7, "level": 0}, SAM: {"id": 8, "level": 1}, ZOMBIE: {"id": 9, "level": 0},
      BROKEN: {"id": 10, "level": 0}, NOBODY: {}}
NO_TOKEN = {"token": False}


def ask(op, uuid, on="server", **hint):
    return {"op": op, "on": on, "uuid": uuid, **{**AT[uuid.lower()], **hint}}


print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())

## The calls of the contract

`okf/api.md` section 6.1 says what the mapping entry of `Player.getAllowFlight()` becomes in the
contract: the call `playerAllowFlight`, on the thread `level`, with the arguments of an entity and
an answer of one boolean. The cell reads every file of `bridge/contract/`, which are written by
hand until a generator writes them, and holds that call to the page. Every call names its thread,
its arguments and its answer and nothing else, every value of an answer has a name and a type,
and every call has its function in the module of its area in `golem-game` and its method in the
seam that answers the calls, each under the name of the call in snake case.

In [ ]:
def snake(name):
    return re.sub(r"(?<!^)([A-Z])", r"_\1", name).lower()


VALUE = re.compile(r"^[a-z][A-Za-z0-9]*: (boolean|byte|short|int|long|float|double)$")
contract = {}
for path in sorted((P.root / "bridge" / "contract").glob("*.toml")):
    table = tomllib.loads(path.read_text(encoding="utf-8"))
    assert set(table) == {"calls"}, "%s holds more than calls: %s" % (P.rel(path), sorted(table))
    for name, call in table["calls"].items():
        assert name not in contract, "the call %s stands in two files" % name
        assert set(call) == {"thread", "arguments", "answer"}, "%s: %s" % (name, sorted(call))
        assert call["thread"] in ("level", "global", "any", "snapshot", "worldgen"), (name, call["thread"])
        assert call["answer"] and all(VALUE.match(value) for value in call["answer"]), \
            "the answer of %s names a value in another form: %s" % (name, call["answer"])
        contract[name] = {**call, "area": path.stem}
for name, call in contract.items():
    print("%-20s %-10s thread %-6s arguments %-10s answer %s" % (name, call["area"], call["thread"],
                                                               call["arguments"], call["answer"]))

assert contract["playerAllowFlight"] == {"thread": "level", "arguments": ["entity"],
                                         "answer": ["allowFlight: boolean"],
                                         "area": "players"}, contract["playerAllowFlight"]
assert "`playerAllowFlight`, thread `level`, the arguments of an entity, an answer of one boolean" in page

game = P.root / "crates" / "golem-game" / "src"
seam = (game / "seam.rs").read_text(encoding="utf-8")
methods = seam.split("pub trait Contract {", 1)[1].split("\n}\n", 1)[0]
for name, call in contract.items():
    area = (game / "contract" / (call["area"] + ".rs")).read_text(encoding="utf-8")
    assert re.search(r"^pub fn %s\(\s*server: &Server<'_>," % snake(name), area, re.M), \
        "contract/%s.rs holds no function of the call %s" % (call["area"], name)
    assert "/// `%s`: " % name in area, "the function of %s does not name its call" % name
    assert re.search(r"^    fn %s\(&self, " % snake(name), methods, re.M), "the seam holds no method of %s" % name
assert len(re.findall(r"^    fn \w+\(&self", methods, re.M)) == len(contract), "the seam holds a method of no call"

## A call against the scripted contract

The function of a call takes the server token, which a thread holds while a frame is open on it.
In the design a frame stands for one native call of the bridge mod on the thread of a level. No
line of the mod opens one yet, so the devtool opens every frame here. Inside a frame `entityFind`
answers the runtime id, the level and whether the entity is a player, for a UUID in any case, and
`playerAllowFlight` answers whether the player may fly. It answers the same for a name whose
runtime id and level are stale: the script finds an entity by its UUID alone, which is the rule
that the mod has to meet for a hint that misses. A call misses for a UUID that no entity carries,
and `playerAllowFlight` misses for an entity that is no player. A read that throws is a fault with
what the script threw. On the main thread with no frame open there is no token, and no call is
made. A thread of its own gets no token either while a frame is open on the main thread, so the
frames of one thread are not those of another. The count of the calls that the scripted contract
answered shows that no step without a token reached it.

The devtool reads the name of a call before it asks for a token, so a call that the contract does
not hold ends it with a line that says so, wherever the step runs. So does a verb that the
subcommand lacks, and a script that names one UUID for two entities, since two names with one
UUID are one entity, whatever their runtime ids and levels are.

In [ ]:
steps = [
    ask("entityFind", ALEX), ask("entityFind", ALEX.upper()), ask("entityFind", ZOMBIE), ask("entityFind", NOBODY),
    ask("entityFind", BROKEN),
    ask("playerAllowFlight", ALEX), ask("playerAllowFlight", SAM), ask("playerAllowFlight", ALEX, id=99, level=3),
    ask("playerAllowFlight", ZOMBIE), ask("playerAllowFlight", NOBODY), ask("playerAllowFlight", BROKEN),
    ask("playerAllowFlight", ALEX, on="idle"), ask("playerAllowFlight", ALEX, on="thread"),
    ask("entityFind", ALEX, on="thread"),
]
got, calls, inner_calls = answers("contract", "call", name="calls", steps=steps)
for step, answer in zip(steps, got):
    print("%-18s %-7s %-38s %s" % (step["op"], step["on"], step["uuid"], json.dumps(answer)))
alex = {"value": {"id": 7, "level": 0, "player": True}}
assert got[0] == alex and got[1] == alex, got[:2]
assert got[2] == {"value": {"id": 9, "level": 0, "player": False}}, got[2]
assert got[3] == {"miss": True}, got[3]
assert got[4] == {"fault": FAULT}, got[4]
assert got[5] == {"value": True} and got[6] == {"value": False}, got[5:7]
assert got[7] == {"value": True}, "a stale runtime id lost the player: %s" % got[7]
assert got[8] == {"miss": True} and got[9] == {"miss": True}, got[8:10]
assert got[10] == {"fault": FAULT}, got[10]
assert got[11:] == [NO_TOKEN] * 3, "a thread without a frame reached the contract: %s" % got[11:]
print("calls:", calls)
assert (calls, inner_calls) == (11, 0), "the scripted contract answered %d calls, and 11 steps hold a token" % calls

error = refused("contract", "call", name="no-call", document={"steps": [ask("playerFly", ALEX)]})
assert error == "golem-devtool: contract call: the contract has no call playerFly", error
for on in ("idle", "thread"):
    error = refused("contract", "call", name="no-call-" + on, document={"steps": [ask("playerFly", ALEX, on=on)]})
    assert error == "golem-devtool: contract call: the contract has no call playerFly", (on, error)
_, error = devtool("contract", expect=1)
assert error.strip() == "golem-devtool: contract: contract takes the verb call and one file", error
_, error = devtool("contract", "run", write("verb.json", {"steps": []}), expect=1)
assert error.strip() == "golem-devtool: contract: contract has no verb run", error
twice = [{"uuid": ALEX, "id": 7}, {"uuid": ALEX.upper(), "id": 8, "level": 1}]
error = refused("contract", "call", name="twice", document={"entities": twice, "steps": []})
assert error == "golem-devtool: contract call: the script names the UUID %s twice" % ALEX, error
error = refused("contract", "call", name="twice-inner", document={"inner": twice, "steps": []})
assert error == "golem-devtool: contract call: the script names the UUID %s twice" % ALEX, error

## The text of a UUID

A UUID is read in its usual form alone, five groups of 8, 4, 4, 4 and 12 hexadecimal digits joined
by dashes, in any case, and it is printed in that form in lower case. The cell above read Alex in
upper case, and the cell on the coordinate function prints the zombie, whose every group starts
with a zero. This cell hands the devtool texts that are no UUID: a name, the empty text, four
groups, six groups, a group with a letter that is no digit, a sign in place of a digit, a blank in
place of a digit, a blank before and after, no dashes, and a dash one place early. Each ends the
devtool with a line that names the text.

In [ ]:
NO_UUID = [
    "alex", "", ALEX[:23], ALEX + "-0000", ALEX[:-1] + "g", "+" + ALEX[1:], ALEX[:7] + " " + ALEX[8:],
    " " + ALEX, ALEX + " ", ALEX.replace("-", ""), ALEX[:7] + "-" + ALEX[7] + ALEX[9:],
]
assert len(set(NO_UUID)) == len(NO_UUID) and ALEX not in NO_UUID, NO_UUID
for index, text in enumerate(NO_UUID):
    error = refused("contract", "call", name="no-uuid-%d" % index, document={"steps": [{"op": "entityFind", "uuid": text}]})
    print("%-40r %s" % (text, error))
    assert error == ("golem-devtool: contract call: %s is no UUID" % text).strip(), (text, error)

## The server token and its frames

A token belongs to the frame it was taken in, and the token that a thread answers is that of its
innermost frame. The steps of this cell open more than one frame.

- `nested` asks in a frame over the first script, then in a frame over the second script that is
  opened inside the first, then in the first again once the second is closed. The second script
  holds Alex with another runtime id, on another level and with no right to fly, so the three
  answers show which frame answered.
- `stale` takes the token in a frame, closes the frame and makes the call with the token it kept,
  first with no frame open and then while a frame over the second script is open in its place.
  Both calls are refused and never made, since a token is held to the serial of its frame.
- `unordered` opens a frame over the first script and one over the second inside it, takes the
  token there, and drops the outer frame first. Frames close in order, so that drop closes both:
  the thread then answers no token, and the token of the inner frame is refused.
- A plain step after them shows that the frames of the thread are sound again, and a step with no
  frame that none was left open.
- `ending` asks on a thread of its own while that thread ends, in the destructor of a thread
  local that was made before the frames of the thread, so the frames are gone when it asks. It
  gets no token, and the process goes on. A panic there would end the process.

The counts of both scripted contracts show which calls were made.

In [ ]:
steps = [
    ask("playerAllowFlight", ALEX, on="nested"), ask("entityFind", ALEX, on="nested"),
    ask("playerAllowFlight", ALEX, on="stale"), ask("entityFind", ALEX, on="stale"),
    ask("playerAllowFlight", ALEX, on="unordered"),
    ask("playerAllowFlight", ALEX), ask("playerAllowFlight", ALEX, on="idle"),
    ask("playerAllowFlight", ALEX, on="ending"), ask("entityFind", ALEX, on="ending"),
]
got, calls, inner_calls = answers("contract", "call", name="frames", steps=steps)
for step, answer in zip(steps, got):
    print("%-18s %-10s %s" % (step["op"], step["on"], json.dumps(answer)))
assert got[0] == {"before": {"value": True}, "inside": {"value": False}, "after": {"value": True}}, got[0]
inner_alex = {"value": {"id": 70, "level": 2, "player": True}}
assert got[1] == {"before": alex, "inside": inner_alex, "after": alex}, got[1]
OUTLIVED = {"closed": NO_TOKEN, "replaced": NO_TOKEN}
assert got[2] == OUTLIVED and got[3] == OUTLIVED, "a token outlived its frame: %s" % got[2:4]
assert got[4] == {"fresh": NO_TOKEN, "kept": NO_TOKEN}, "an outer frame closed without the frame inside it: %s" % got[4]
assert got[5] == {"value": True} and got[6] == NO_TOKEN, got[5:7]
assert got[7] == NO_TOKEN and got[8] == NO_TOKEN, got[7:]
print("calls:", calls, "of the second script:", inner_calls)
assert (calls, inner_calls) == (5, 2), "the two nested steps make 4 and 2 calls and the plain step 1: %s" % (
    (calls, inner_calls),)

## The coordinate function

The function of a coordinate holds its meaning, so that every way into the game runs the same
code. Today the devtool is its one caller. `player::allow_flight` answers the value of the call. A
miss is the error `ENTITY_GONE`, the code that `okf/api.md` section 6.1 names for a miss. An
entity of another kind, as the zombie that is asked as a player, misses by the rule of the script,
which the page does not state, and the message of the error is true of both. A fault is the error
`GAME_FAULT` with what the script threw. A thread that holds no server token gets
`NOT_ON_SERVER_THREAD`, with the thread that the call needs named in its message, and makes no
call, and so do a call with a token whose frame is closed and a thread that ends. The cell holds
the page to the three codes. `entity::find` answers the name of an entity, which it builds from
the UUID it was asked and from what the call answered, and the object type that the entity answers
as, `Player` or `EntityObject`. It answers none for a UUID that no entity carries.
`server::bukkit_version` is a coordinate of the thread class `any`: it takes no token, so it
answers on the main thread inside a frame and outside one, on a thread of its own and on a thread
that ends, and it makes no call. Its value is the one that `okf/api.md` section 9.1 names. Each
other coordinate that answers makes exactly one call.

In [ ]:
steps = [
    ask("player::allow_flight", ALEX), ask("player::allow_flight", SAM), ask("player::allow_flight", ZOMBIE),
    ask("player::allow_flight", NOBODY), ask("player::allow_flight", BROKEN),
    ask("player::allow_flight", ALEX, on="idle"), ask("player::allow_flight", ALEX, on="thread"),
    ask("entity::find", ALEX), ask("entity::find", ALEX.upper()), ask("entity::find", ZOMBIE),
    ask("entity::find", NOBODY), ask("entity::find", BROKEN), ask("entity::find", ALEX, on="thread"),
    ask("player::allow_flight", ALEX, on="stale"), ask("entity::find", ALEX, on="stale"),
    ask("player::allow_flight", ALEX, on="ending"), ask("entity::find", ALEX, on="nested"),
    *({"op": "server::bukkit_version", "on": on} for on in ("server", "idle", "thread", "ending")),
]
got, calls, inner_calls = answers("api", "coordinate", name="coordinates", steps=steps)
for step, answer in zip(steps, got):
    print("%-24s %-7s %-38s %s" % (step["op"], step["on"], step.get("uuid", ""), json.dumps(answer)))
GONE = {"error": "ENTITY_GONE", "message": "The entity is gone or is of another kind."}
BROKE = {"error": "GAME_FAULT", "message": "The game gave no answer: " + FAULT}
assert got[0] == {"value": True} and got[1] == {"value": False}, got[:2]
assert got[2] == GONE and got[3] == GONE, got[2:4]
assert got[4] == BROKE, got[4]
off_thread = got[5]
assert off_thread["error"] == "NOT_ON_SERVER_THREAD" and "server thread" in off_thread["message"], off_thread
assert got[6] == off_thread, got[6]
found_alex = {"value": {"id": 7, "level": 0, "uuid": ALEX, "kind": "Player"}}
assert got[7] == found_alex and got[8] == found_alex, got[7:9]
assert got[9] == {"value": {"id": 9, "level": 0, "uuid": ZOMBIE, "kind": "EntityObject"}}, got[9]
assert got[10] == {"value": None}, got[10]
assert got[11] == BROKE, got[11]
assert got[12] == off_thread, got[12]
outlived = {"closed": off_thread, "replaced": off_thread}
assert got[13] == outlived and got[14] == outlived, "a token outlived its frame: %s" % got[13:15]
assert got[15] == off_thread, got[15]
assert got[16] == {"before": found_alex, "inside": {"value": {"id": 70, "level": 2, "uuid": ALEX, "kind": "Player"}},
                   "after": found_alex}, got[16]
version = got[17]["value"]
assert got[17:] == [{"value": version}] * 4, got[17:]
assert "`bukkitVersion` (the value of spigot-api, `%s`)" % version in page, version
print("calls:", calls, "of the second script:", inner_calls)
assert (calls, inner_calls) == (12, 1), "ten plain steps reach the script, and the nested step makes 2 and 1: %s" % (
    (calls, inner_calls),)
for code, sentence in (("ENTITY_GONE", "with `ENTITY_GONE` on a miss"),
                       ("GAME_FAULT", "is the error `GAME_FAULT` of its coordinate"),
                       ("NOT_ON_SERVER_THREAD", "or the error `NOT_ON_SERVER_THREAD` on any other thread")):
    assert sentence in page, "okf/api.md does not name the code %s in the words: %s" % (code, sentence)

error = refused("api", "coordinate", name="no-function", document={"steps": [ask("player::fly", ALEX, on="thread")]})
assert error == "golem-devtool: api coordinate: golem-model has no coordinate function player::fly", error
_, error = devtool("api", expect=1)
assert error.strip() == "golem-devtool: api: api takes the verb coordinate and one file", error
_, error = devtool("api", "sdl", write("verb.json", {"steps": []}), expect=1)
assert error.strip() == "golem-devtool: api: api has no verb sdl", error

## Verdict

The cell clears the temporary directory. A run that reaches it passed every section.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: the calls of the contract match okf/api.md, a call answers the value, the miss and the fault of "
      "the script, no call is made without an open frame of the calling thread, a coordinate answers the "
      "codes that the page names for a miss, a failed call and a thread without a token, and a coordinate of "
      "the class any answers on every thread")